# Measuring RPKI and BGPsec Deployment in the Wild

Origin validation works, path validation works, and MANRS exists to get both adopted. None
of that answers the question an operator — or an attacker — actually cares about: **how
much of the real internet uses any of it?**

In this notebook you will measure deployment yourself, from public data, instead of taking
anyone's word for it:

- load one day's view of **every announced IPv4 prefix** on the internet, and the full set
  of **VRPs** for the same day
- run **ROV over the whole table** — about 1.1 million announcements — and measure how much
  of the internet RPKI actually vouches for
- compare **ROA publishing between MANRS participants and everyone else**: does a public
  commitment show up in data the participants don't control?
- count how many networks have taken even the **first deployable step of BGPsec**

---
## 1. Three questions, three datasets

| question | dataset | source |
| --- | --- | --- |
| What is **authorized**? | Validated ROA payloads (VRPs) from all five trust anchors | RIPE NCC daily archive |
| What is **announced**? | Routeviews prefix-to-AS (pfx2as): every announced IPv4 prefix and its observed origin AS | CAIDA public data archive |
| Who has **publicly committed**? | The ASNs of approved MANRS participants | MANRS Observatory public API |

The routing table and the RPKI snapshot are pinned to the same archived day,
`SNAPSHOT_DATE`, so the comparison is internally consistent — both archives publish daily,
and any recent day works. The MANRS participant list is different: it is fetched **live**,
because no public historical archive of it exists. Keep that mismatch in mind; you will
come back to it in the reflection.

In [1]:
%pip install pandas requests matplotlib

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached matplotlib-3.11.1-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp312-cp312-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 9.8/9.8 MB 101.8 MB/s eta 0:00:00
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached matplotlib-3.11.1-cp31


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gzip
import io
import ipaddress
import json
import lzma
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

# The snapshot day to analyze. Both archives publish daily; any recent day works.
SNAPSHOT_DATE = "2026-07-01"

# The five RPKI trust anchors, one per RIR.
TRUST_ANCHORS = ["afrinic", "ripencc", "arin", "lacnic", "apnic"]

VRP_ARCHIVE_URL = "https://ftp.ripe.net/ripe/rpki/{ta}.tal/{yyyy}/{mm}/{dd}/roas.csv.xz"
PFX2AS_URL = (
    "https://publicdata.caida.org/datasets/routing/routeviews-prefix2as/"
    "{yyyy}/{mm}/routeviews-rv2-{yyyy}{mm}{dd}-1200.pfx2as.gz"
)
MANRS_ASNS_URL = "https://observatory.manrs.org/api/v2/asns"
RPKI_CLIENT_URL = "https://console.rpki-client.org/rpki.json"

# Downloaded files are cached here so re-running the notebook doesn't re-fetch them.
CACHE_DIR = Path("deployment_cache")
CACHE_DIR.mkdir(exist_ok=True)

print(f"Snapshot date: {SNAPSHOT_DATE}")

Snapshot date: 2026-07-01


### 1.1 What is authorized — the day's VRPs

RIPE NCC publishes a daily historical archive of validated ROA payloads covering **all
five** trust anchors — one CSV per trust anchor per day:

```
https://ftp.ripe.net/ripe/rpki/<ta>.tal/<YYYY>/<MM>/<DD>/roas.csv.xz

<ta> is one of:  afrinic, ripencc, arin, lacnic, apnic
```

`load_rpki_snapshot()` downloads the five files, normalizes them, and returns one row per
VRP:

| field | example | description |
| --- | --- | --- |
| `prefix` | `137.110.0.0/16` | prefix authorized by the ROA (normalized) |
| `max_length` | `20` | most specific announcement allowed |
| `asn` | `7377` | AS authorized to originate the prefix |
| `trust_anchor` | `arin` | RPKI trust anchor that validated the VRP |
| `family` | `4` | IP address family (4 or 6) |
| `date` | `2026-07-01` | snapshot date the rows were loaded for |

In [3]:
def _fetch_vrp_csv(trust_anchor: str, date: str) -> bytes:
    """Download (or read from cache) one trust anchor's VRP file for one day."""
    yyyy, mm, dd = date.split("-")
    cache_path = CACHE_DIR / f"{trust_anchor}-{date}.csv"

    if cache_path.exists():
        return cache_path.read_bytes()

    url = VRP_ARCHIVE_URL.format(ta=trust_anchor, yyyy=yyyy, mm=mm, dd=dd)
    response = requests.get(url, timeout=180)
    response.raise_for_status()

    raw = lzma.decompress(response.content)
    cache_path.write_bytes(raw)
    return raw


def _normalize(df: pd.DataFrame, trust_anchor: str, date: str) -> pd.DataFrame:
    """Map RIPE's CSV columns onto the schema described above."""
    # RIPE's header has varied over the years; match it case-insensitively.
    lookup = {c.strip().lower(): c for c in df.columns}
    asn_col = lookup.get("asn")
    prefix_col = lookup.get("ip prefix", lookup.get("prefix"))
    maxlen_col = lookup.get("max length", lookup.get("maxlength"))

    out = pd.DataFrame()
    out["prefix"] = df[prefix_col].astype(str).str.strip()

    # ASNs appear as "AS7377" or "7377" depending on the vintage of the file.
    out["asn"] = (
        df[asn_col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"^AS", "", regex=True)
        .astype(int)
    )

    # A blank max length means "exactly the prefix's own length".
    prefix_len = out["prefix"].str.split("/").str[1].astype(int)
    out["max_length"] = (
        pd.to_numeric(df[maxlen_col], errors="coerce").fillna(prefix_len).astype(int)
    )

    out["trust_anchor"] = trust_anchor
    out["family"] = out["prefix"].str.contains(":").map({True: 6, False: 4})
    out["date"] = date

    return out[["prefix", "max_length", "asn", "trust_anchor", "family", "date"]]


def load_rpki_snapshot(date: str = SNAPSHOT_DATE, trust_anchors=TRUST_ANCHORS) -> pd.DataFrame:
    """Return one row per VRP across all trust anchors for the given day."""
    frames = []
    for ta in trust_anchors:
        raw = _fetch_vrp_csv(ta, date)
        df = pd.read_csv(io.BytesIO(raw))
        frames.append(_normalize(df, ta, date))
        print(f"  {ta:<8} {len(frames[-1]):>9,} VRPs")

    return pd.concat(frames, ignore_index=True)

In [4]:
print(f"Loading VRPs for {SNAPSHOT_DATE} ...")
vrps = load_rpki_snapshot()

# Sanity check: by mid-2026 the five trust anchors together publish close to a
# million VRPs, with ripencc, apnic, and arin the three largest.
print(f"\nTotal: {len(vrps):,} VRPs")
vrps.head()

Loading VRPs for 2026-07-01 ...
  afrinic     28,937 VRPs
  ripencc    348,596 VRPs
  arin       260,483 VRPs
  lacnic      42,605 VRPs
  apnic      287,913 VRPs

Total: 968,534 VRPs


,prefix,max_length,asn,trust_anchor,family,date
0,41.0.0.0/16,24,36994,afrinic,4,2026-07-01
1,41.10.0.0/16,24,36994,afrinic,4,2026-07-01
2,41.21.128.0/17,24,36994,afrinic,4,2026-07-01
3,41.23.0.0/16,24,36994,afrinic,4,2026-07-01
4,41.24.0.0/16,24,36994,afrinic,4,2026-07-01


### 1.2 What is announced — the day's routing table

CAIDA's **Routeviews prefix-to-AS** (`pfx2as`) dataset condenses a day of global BGP
routing-table snapshots into exactly the thing ROV cares about: every announced prefix,
together with the origin AS observed announcing it.

```
https://publicdata.caida.org/datasets/routing/routeviews-prefix2as/
        <YYYY>/<MM>/routeviews-rv2-<YYYYMMDD>-1200.pfx2as.gz
```

Each non-comment row is tab-separated:

| position | field | example | description |
| --- | --- | --- | --- |
| 1 | network | `35.228.224.0` | network address |
| 2 | prefix length | `19` | combine with the network: `35.228.224.0/19` |
| 3 | origin AS | `15169` | observed origin AS |

Two details the loader below handles for you:

- **Multi-origin rows.** Some prefixes are observed with several origins
  (`203.0.113.0  24  64500_64501`). Each origin becomes its own row — ROV classifies
  **(prefix, origin) pairs**, so a prefix with two origins is two separate questions.
- **IPv4 only.** The `rv2` files cover IPv4. (A companion Routeviews IPv6 dataset
  exists; everything in this notebook would work the same way on it.)

In [5]:
def load_prefix2as_snapshot(date: str = SNAPSHOT_DATE) -> pd.DataFrame:
    """Return one row per announced (prefix, origin) pair observed on the given day."""
    yyyy, mm, dd = date.split("-")
    cache_path = CACHE_DIR / f"pfx2as-{date}.csv.gz"

    if cache_path.exists():
        return pd.read_csv(cache_path)

    url = PFX2AS_URL.format(yyyy=yyyy, mm=mm, dd=dd)
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    text = gzip.decompress(response.content).decode("utf-8", errors="ignore")

    rows = []
    for line in text.splitlines():
        fields = line.split()
        if len(fields) < 3:
            continue
        prefix = f"{fields[0]}/{fields[1]}"
        # Multi-origin rows list several ASes ("64500_64501"); one row per origin.
        for token in re.split(r"[_,{}]+", fields[2]):
            if token.isdigit():
                rows.append((prefix, int(token)))

    bgp = pd.DataFrame(rows, columns=["prefix", "origin"]).drop_duplicates(ignore_index=True)
    bgp["date"] = date
    bgp.to_csv(cache_path, index=False, compression="gzip")
    return bgp

In [6]:
bgp = load_prefix2as_snapshot()

# Sanity check: by mid-2026 you should see roughly 1.1 million pairs.
print(f"{len(bgp):,} announced (prefix, origin) pairs on {SNAPSHOT_DATE}")
bgp.head()

1,107,180 announced (prefix, origin) pairs on 2026-07-01


,prefix,origin,date
0,1.0.0.0/24,13335,2026-07-01
1,1.0.4.0/24,38803,2026-07-01
2,1.0.5.0/24,38803,2026-07-01
3,1.0.6.0/24,38803,2026-07-01
4,1.0.7.0/24,38803,2026-07-01


### 1.3 Who has publicly committed — MANRS participants

The MANRS Observatory exposes a small public endpoint that returns the ASNs claimed by
**approved MANRS participants** across all of its programs:

```
https://observatory.manrs.org/api/v2/asns
```

An ASN on this list belongs to an operator that applied to MANRS, committed to the
actions — filtering, anti-spoofing, coordination, and global validation — and was
approved. This notebook holds the list to the most checkable of those commitments:
participants promised, under Action 4, to publish ROAs. **Did they?**

> **Helpful hint:** the endpoint returns the *current* membership, not membership as of
> `SNAPSHOT_DATE`. The result is cached under `deployment_cache/`, so re-runs keep using
> the list from the day you first ran this cell.

In [7]:
def load_manrs_asns() -> set:
    """Return the set of ASNs claimed by approved MANRS participants (fetched live)."""
    cache_path = CACHE_DIR / "manrs_asns.json"

    if cache_path.exists():
        return set(json.loads(cache_path.read_text()))

    response = requests.get(MANRS_ASNS_URL, timeout=60)
    response.raise_for_status()
    asns = response.json()["asns"]
    cache_path.write_text(json.dumps(asns))
    return set(asns)


manrs_asns = load_manrs_asns()
print(f"{len(manrs_asns):,} ASNs are claimed by approved MANRS participants")

1,452 ASNs are claimed by approved MANRS participants


---
## 2. ROV at internet scale

Classifying one announcement is the same three-step ROV lookup — **Valid**, **Invalid**,
or **Not Found**. The problem here is scale: ~1.1 million announcements against ~1
million VRPs is far too many for a scan of the VRP table per announcement, so the VRPs
are indexed by exact prefix instead.

`build_vrp_index()` and `covering_vrps()` below are provided — read them before building
on top of them. How the index works, a worked masking example, and implementation
guidance are under Task 4 in [Tasks 4–7](Task-deployment-exercise.md).

In [8]:
def build_vrp_index(vrps_df: pd.DataFrame):
    """Index IPv4 VRPs by (network address, prefix length) for fast covering lookups.

    Returns (table, lengths):
      table   maps (network_int, length) -> list of (vrp_prefix, asn, max_length)
      lengths is the sorted list of prefix lengths that occur in the table
    """
    v4 = vrps_df[vrps_df["family"] == 4]

    table = {}
    for prefix, asn, max_length in zip(v4["prefix"], v4["asn"], v4["max_length"]):
        network, _, length = prefix.partition("/")
        key = (int(ipaddress.IPv4Address(network)), int(length))
        table.setdefault(key, []).append((prefix, int(asn), int(max_length)))

    lengths = sorted({length for _, length in table})
    return table, lengths


def covering_vrps(prefix: str, vrp_index) -> list:
    """Return every VRP whose prefix covers `prefix` (IPv4 only).

    Each element is a tuple (vrp_prefix, asn, max_length).
    """
    table, lengths = vrp_index

    network, _, length = prefix.partition("/")
    address = int(ipaddress.IPv4Address(network))
    prefix_length = int(length)

    covering = []
    for vrp_length in lengths:
        if vrp_length > prefix_length:
            break  # a longer prefix can never cover a shorter one
        mask = (0xFFFFFFFF << (32 - vrp_length)) & 0xFFFFFFFF if vrp_length else 0
        hit = table.get((address & mask, vrp_length))
        if hit:
            covering.extend(hit)
    return covering

In [9]:
vrp_index = build_vrp_index(vrps)

table, lengths = vrp_index
print(f"Indexed {len(table):,} distinct IPv4 VRP prefixes")
print(f"Prefix lengths present: /{lengths[0]} through /{lengths[-1]} "
      f"({len(lengths)} distinct lengths)")

Indexed 651,345 distinct IPv4 VRP prefixes
Prefix lengths present: /8 through /32 (25 distinct lengths)


### Task 4: implement `classify_rov()`

Fill in `classify_rov()` below so it returns `"Valid"`, `"Invalid"`, or `"Not Found"`
for one announcement, using `covering_vrps()` for step 1. Read the Task 4 guidance in
[Tasks 4–7](Task-deployment-exercise.md) before writing code.

In [ ]:
def classify_rov(prefix: str, origin_asn: int, vrp_index) -> str:
    """Classify one announced (prefix, origin) pair against the VRP index.

    Args:
        prefix:     the announced prefix, e.g. "137.110.0.0/16"
        origin_asn: the AS observed originating it, e.g. 7377
        vrp_index:  the (table, lengths) structure from build_vrp_index()

    Returns:
        One of "Valid", "Invalid", or "Not Found".
    """
    # TODO: implement the three-step lookup (see Task 4 in Task-deployment-exercise.md).

    raise NotImplementedError("classify_rov is yours to write")

### Check yourself on a hand-built table

Before touching a million real announcements, make sure every case below passes.

In [11]:
TEST_VRPS = pd.DataFrame(
    [
        # prefix,           max_length, asn,   trust_anchor, family, date
        ("203.0.113.0/24", 24, 64500, "arin", 4, "test"),
        ("198.51.100.0/24", 26, 64501, "ripencc", 4, "test"),
        ("192.0.2.0/24", 24, 64502, "apnic", 4, "test"),
        ("192.0.2.0/24", 24, 64503, "lacnic", 4, "test"),  # same prefix, second origin
    ],
    columns=["prefix", "max_length", "asn", "trust_anchor", "family", "date"],
)
TEST_INDEX = build_vrp_index(TEST_VRPS)

TEST_CASES = [
    # (prefix, origin, expected, what it exercises)
    ("203.0.113.0/24", 64500, "Valid", "exact match on prefix and origin"),
    ("203.0.113.0/24", 66000, "Invalid", "covered, wrong origin"),
    ("203.0.113.128/25", 64500, "Invalid", "right origin, beyond max length"),
    ("198.51.100.192/26", 64501, "Valid", "more specific, but within max length"),
    ("198.51.100.192/28", 64501, "Invalid", "more specific than max length allows"),
    ("192.0.2.0/24", 64503, "Valid", "second covering VRP authorizes it"),
    ("10.0.0.0/8", 64500, "Not Found", "no VRP covers this prefix at all"),
]

passed = 0
for prefix, origin, expected, description in TEST_CASES:
    actual = classify_rov(prefix, origin, TEST_INDEX)
    ok = actual == expected
    passed += ok
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {prefix:<18} AS{origin:<6} got {actual:<9} "
          f"expected {expected:<9} ({description})")

print(f"\n{passed}/{len(TEST_CASES)} cases passed")

NotImplementedError: classify_rov is yours to write

### Classify all 1.1 million announcements (provided)

`annotate_rov()` runs your classifier over the whole table and caches the result — this
cell will take a minute or two to run the first time

In [ ]:
def annotate_rov(bgp_df: pd.DataFrame, vrp_index, date: str = SNAPSHOT_DATE,
                 force_refresh: bool = False) -> pd.DataFrame:
    """Return a copy of bgp_df with a rov_state column, cached on disk."""
    cache_path = CACHE_DIR / f"rov-annotated-{date}.csv.gz"
    if cache_path.exists() and not force_refresh:
        return pd.read_csv(cache_path)

    states = []
    for count, (prefix, origin) in enumerate(zip(bgp_df["prefix"], bgp_df["origin"]), 1):
        states.append(classify_rov(prefix, int(origin), vrp_index))
        if count % 250_000 == 0:
            print(f"  classified {count:>9,} / {len(bgp_df):,}")

    annotated = bgp_df.copy()
    annotated["rov_state"] = states
    annotated.to_csv(cache_path, index=False, compression="gzip")
    return annotated


bgp_classified = annotate_rov(bgp, vrp_index)
bgp_classified.head()

---
## 3. How much of the internet implements RPKI? (Task 5)

The first deployment question is the bluntest: across everything announced on
`SNAPSHOT_DATE`, how much does RPKI vouch for? From `bgp_classified`, produce the
per-state count/percentage table and the bar chart of shares specified under Task 5 in
[Tasks 4–7](Task-deployment-exercise.md).

In [ ]:
# Task 5: internet-wide ROV state shares — the counts table, then the bar chart.
# See Task 5 in Task-deployment-exercise.md.
#
# YOUR CODE HERE
raise NotImplementedError

**Q1.** Report your headline numbers: what share of announced (prefix, origin) pairs is
**Valid**, **Not Found**, and **Invalid**? Then two follow-ups: (a) operators who enforce
ROV drop Invalid routes — so who is actually affected by the Invalid share you measured,
and why is that share so much smaller than the other two? (b) What does the Not Found
share say about how much of the internet would go dark if operators dropped Not Found
routes today?

*Write your answer here.*

---
## 4. ROA coverage, one network at a time (Task 6)

Deployment is a **per-operator decision**, so the more telling unit is the origin AS:
what fraction of *its own* announcements does RPKI vouch for? Build the `per_as`
DataFrame and plot its `valid_frac` histogram as specified under Task 6 in
[Tasks 4–7](Task-deployment-exercise.md). The next section reuses `per_as`.

In [ ]:
# Task 6: per-AS ROA coverage — build per_as, then plot the valid_frac histogram.
# See Task 6 in Task-deployment-exercise.md.
#
# YOUR CODE HERE
raise NotImplementedError

**Q2.** Describe the shape of the histogram. Most ASes sit at one of the two extremes —
all or nothing — rather than in the middle. Why does ROA publishing, as an operational
decision, produce exactly this shape? What kind of AS would you expect to find *in* the
middle?

*Write your answer here.*

---
## 5. MANRS participants versus everyone else (Task 7)

Now the comparison this notebook exists for: every ASN in `manrs_asns` belongs to an
operator that **publicly committed** to publishing ROAs, and if that commitment means
anything, it should be visible in data the participants don't control. Produce the three
deliverables — the announcement-level crosstab, the per-AS summary table, and the
distribution plot — specified under Task 7 in
[Tasks 4–7](Task-deployment-exercise.md).

In [ ]:
# Task 7: the MANRS comparison — the announcement-level crosstab, the per-AS
# summary table, and the distribution plot. See Task 7 in Task-deployment-exercise.md.
#
# YOUR CODE HERE
raise NotImplementedError

**Q3.** Quantify the gap: how do the two groups compare on `median_valid_frac` and
`share_zero`? Then the harder question — this is an observational comparison, not an
experiment. Give one mechanism by which MANRS participation *causes* better ROA coverage,
and one by which the causality runs the other way (self-selection). What data would help
you tell the two apart?

*Write your answer here.*

---
## 6. And BGPsec? (Task 8)

ROAs measure origin validation's *publish* side. BGPsec has an exactly analogous
deployable artifact: the **router certificate** — the RPKI object binding a router's
public key to its ASN. No router certificate, no verifiable signatures; publishing one is
the first observable step any AS must take. Counting router certificates in the global
RPKI therefore measures how many networks have taken even that first step.

The `rpki-client` project runs a public validator whose full validated output — every ROA,
ASPA record, and BGPsec router key in the global RPKI — is published as one JSON document
at [console.rpki-client.org](https://console.rpki-client.org/). This is **live** data
(validated minutes ago, not on `SNAPSHOT_DATE`), and the full document is ~100 MB — the
loader below keeps only the few numbers we need and caches them, so it downloads once.

In [ ]:
def load_bgpsec_snapshot(force_refresh: bool = False) -> dict:
    """Fetch the live validated-RPKI summary and keep only the small parts we need."""
    cache_path = CACHE_DIR / "bgpsec_snapshot.json"

    if cache_path.exists() and not force_refresh:
        return json.loads(cache_path.read_text())

    print("Downloading the full validated-RPKI snapshot (~100 MB, one time) ...")
    response = requests.get(RPKI_CLIENT_URL, timeout=600)
    response.raise_for_status()
    data = response.json()

    snapshot = {
        "buildtime": data["metadata"]["buildtime"],
        "roas": data["metadata"]["roas"],
        "vrps": data["metadata"]["vrps"],
        "aspas": data["metadata"]["aspas"],
        "bgpsec_keys": data["bgpsec_keys"],
    }
    cache_path.write_text(json.dumps(snapshot, indent=1))
    return snapshot


snapshot = load_bgpsec_snapshot()
router_asns = sorted({key["asn"] for key in snapshot["bgpsec_keys"]})

print(f"Validated RPKI as of {snapshot['buildtime']}\n")
print(f"  ROAs                        {snapshot['roas']:>9,}")
print(f"  VRPs                        {snapshot['vrps']:>9,}")
print(f"  ASPA records                {snapshot['aspas']:>9,}")
print(f"  BGPsec router certificates  {len(snapshot['bgpsec_keys']):>9,}")
print(f"\nASNs with at least one BGPsec router certificate: {router_asns}")

**Q4.** Put the two counts side by side: ROAs versus BGPsec router certificates. Explain
the gap in terms of what a network *gets* for publishing each one today — one is a
unilateral act with immediate benefit, the other buys nothing until neighbors join. Note
the ASPA count as well: ASPA is years younger than BGPsec yet already far ahead of it.
What does that suggest about which kinds of path-security design can actually deploy?

*Write your answer here.*

---
## 7. Reflect

1. **The timing mismatch.** Your MANRS list is from today; the routing table and VRPs are
   from `SNAPSHOT_DATE`. Suppose a network published its first ROAs in the gap between
   the two dates, or joined MANRS last week. Which direction would each case push your
   MANRS comparison — overstate or understate the participants' lead?
2. **The half you cannot see.** This notebook measured *publishing* — MANRS Action 4. The
   other half of origin validation, networks actually *enforcing* ROV on the routes they
   hear (Action 1), is invisible in every dataset you used. Why? Sketch how you would
   measure ROV enforcement from the outside: what would you need to announce, and what
   would you watch for?

*Write your answers here.*

### Where measurement continues

- [MANRS Observatory](https://observatory.manrs.org/) — per-participant conformance scoring, the fuller version of what you just did.
- [NIST RPKI Monitor](https://rpki-monitor.antd.nist.gov/) — long-running RPKI adoption time series to check your numbers against.
- [Is BGP Safe Yet?](https://isbgpsafeyet.com/) — tests whether *your* ISP enforces ROV; one answer to reflection question 2.
- [RoVista](https://rovista.netsecurelab.org/) — measures ROV enforcement across ASes; the research-grade answer to reflection question 2.
- [rpki-client console](https://console.rpki-client.org/) — the live validated-RPKI snapshot the BGPsec count came from.

In [ ]:
# Scratch space.